In [64]:
import rebound
import reboundx
import numpy as np

In [49]:
sim = rebound.Simulation()
# sim.add(m=1.,hash='sun') 
# sim.add(m=1.e-3, a=1., e=0.2, hash='earth')
sim.add('Sun', hash='sun')
# sim.add('Mercury')
# sim.add('Venus')
sim.add('Earth', hash='earth')
# sim.add('Mars')
# sim.add('Jupiter')
# sim.add('Saturn')
# sim.add('Uranus')
# sim.add('Neptune')
sim.move_to_com
sim.convert_particle_units('AU', 'year', 'Msun')
ps = sim.particles

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')


Now we add REBOUNDx and our effect as usual:

In [50]:
sim.dt = np.sqrt(10)/365.25
# sim.dt = 15/365
sim.exit_min_distance = 1e-3 # ~4x the distance between the Earth and Moon.
sim.exit_max_distance = 1000. # 1000 AU, Unlikely that a planet is bound.

In [51]:
rebx = reboundx.Extras(sim)

gr = rebx.load_force('gr_potential')
rebx.add_force(gr)
gr.params['c'] = 63240 # speed of light in AU/yr

cf = rebx.load_force("quadrupole")
rebx.add_force(cf)

earth_m = ps['earth'].m/1.0123000370338813
f = 0.8525
mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)

R = 0.0025696

ps['earth'].params["Rcentral"] = R
ps['earth'].params["mu_effcentral"] = mu_eff

1. compare 30myr integration (again w. 15 AND whfast). test 15 10kyr.
2. compare zeebe (check_int)

In [ ]:
%%time

Nout=300
pomega, Eerr, E, coms = np.zeros(Nout), np.zeros(Nout), np.zeros(Nout), np.zeros(Nout)
times = np.linspace(0,1.e6,Nout)
sim.move_to_com()
E0 = sim.energy() + rebx.quad_force_potential()  + rebx.gr_potential_potential(gr)
print(sim.com())
for i, time in enumerate(times):
    # pomega[i] = ps['earth'].pomega
    sim.integrate(time)
    E[i] = sim.energy()  + rebx.quad_force_potential() + rebx.gr_potential_potential(gr)
    # E[i] = sim.energy()  + rebx.quad_force_potential()
    Eerr[i] = abs((E[i]-E0)/E0)
    com = sim.com()
    coms[i] = np.sqrt(com.x**2 + com.y**2 + com.z**2)

print(sim.com())

<rebound.particle.Particle object at 0x7fd0f9f99450, m=1.000003040427676 x=0.0 y=1.0587879648996438e-22 z=3.231164443663464e-27 vx=0.0 vy=0.0 vz=0.0>


In [ ]:
def theoretical(times):
    y = np.sqrt(times)
    return y

In [ ]:
fig, axarr = plt.subplots(nrows=1, figsize=(10,5))
# fig.title("gr_potential error profile")
axarr.plot(times, Eerr, '.')
axarr.plot(times, 1e-17*theoretical(times), '-',label='Slope=0.5')
axarr.plot(times, 2e-19*times, '-', label='Slope=1')
axarr.set_xscale('log')
axarr.set_yscale('log')
axarr.set_xlabel('Time', fontsize=24)
axarr.set_ylabel('Energy Error', fontsize=24)
axarr.legend()
fig.suptitle("gr_potential error profile")
fig.show()

In [ ]:
fig, axarr = plt.subplots(nrows=1, figsize=(10,5))
axarr.plot(times, coms, '.')
# axarr.set_xscale('log')
# axarr.set_yscale('log')
axarr.set_xlabel('Time', fontsize=24)
axarr.set_ylabel('com', fontsize=24)
# axarr.legend()
fig.suptitle("gr_potential error profile")
fig.show()